# WatchWise 2.0 on Kaggle

Fairness-aware **group** movie recommendation: diffusion-generated compromise candidates vs traditional nearest-neighbour retrieval, picked by a fairness-aware reranker (greedy bandit + REINFORCE slate-builder), grounded in real MovieLens ratings.

**This notebook runs unchanged across every Kaggle accelerator.** Set `ACCELERATOR` in the cell below to match the runtime you picked in *Settings -> Accelerator*:

| Kaggle menu | `ACCELERATOR` value |
| --- | --- |
| None | `"cpu"` |
| GPU T4 x2 | `"cuda_multi"` |
| GPU P100 | `"cuda"` |
| TPU v5e-8 | `"tpu"` |

Or just use `"auto"` and it detects the right one.

In [ ]:
# === 1. Get the code ===
# Option A: add this repo as a Kaggle dataset/utility and point PROJECT at it.
# Option B: clone from GitHub (uncomment).
import os, sys, subprocess
PROJECT = "/kaggle/working/Movie-WatchWise"
# subprocess.run(["git", "clone", "<your-repo-url>", PROJECT])
if not os.path.isdir(PROJECT):
    # fall back to the current directory if the repo is already here
    PROJECT = os.getcwd()
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print("project:", PROJECT)

In [ ]:
# === 2. Dependencies (Kaggle already ships torch/pandas/sklearn) ===
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio>=4,<5", "plotly", "sentence-transformers", "pyarrow"])
# TPU only: torch_xla is preinstalled on Kaggle's TPU image.

In [ ]:
# === 3. Choose the accelerator to match your Kaggle runtime ===
ACCELERATOR = "auto"   # "cpu" | "cuda" (P100) | "cuda_multi" (T4 x2) | "tpu" | "auto"
PHASE = "phase1"        # "phase1" = ml-latest-small (fast); "phase2" = ml-25m (scale)

from watchwise.accelerator import Accelerator
print(Accelerator(ACCELERATOR))

In [ ]:
# === 4. Run the whole pipeline (download -> MF -> diffusion -> groups -> RL -> eval) ===
import runpy
for stage in ["01_prepare_data", "02_train_mf", "03_train_diffusion",
              "04_form_groups", "05_train_rl", "06_evaluate"]:
    print(f"\n########## {stage} ##########")
    sys.argv = [stage, "--phase", PHASE, "--accelerator", ACCELERATOR]
    runpy.run_path(f"scripts/{stage}.py", run_name="__main__")

In [ ]:
# === 5. Inspect the headline result ===
import pandas as pd
from IPython.display import Markdown, display
comp = pd.read_csv("results/comparison.csv")
display(comp[comp.kind == "divergent"][["method", "relevance", "min_member_sat",
       "fairness_gap", "group_ndcg", "group_hit", "diversity"]].round(3))
display(Markdown(open("results/summary.md").read()))

In [ ]:
# === 6. Figures ===
from IPython.display import Image
for f in ["fig_min_member_by_method.png", "fig_candidate_generation.png",
          "fig_w2_sweep.png", "fig_min_member_by_kind.png"]:
    display(Image(f"results/{f}"))

In [ ]:
# === 7. (Optional) launch the Gradio demo with a public share link ===
from app.demo import DemoEngine, build_ui
build_ui(DemoEngine(PHASE)).launch(share=True)